In [1]:
import numpy as np
from tqdm import tqdm

## 1. Tutorial: How to use PatientSpace

**Overview**

In this notebook, we compare the results of our model's inference against PatientSpace graph prediction.

Prerequisite: You should have already run the CLI inference to generate your precomputed results:

```bash
python -m inference --t1 ./data/subject_01_T1.nii.gz --pet ./data/subject_01_PET.nii.gz --sd output
```

## 2. Load and compute PatientSpace Graph

In this section, we build and predict graph predicton

### Graphe creation

In [2]:
def KL_DIV(q_mu, q_logvar, p_mu, p_logvar):
    return 0.5 * np.sum(p_logvar - q_logvar - 1 + (q_mu - p_mu)**2 / np.exp(p_logvar) + np.exp(q_logvar) / np.exp(p_logvar), axis =-1)

In [3]:
patientspace_mu = np.load("precomputed/precomputed_mu_patientspace.npy")
patientspace_logvar = np.load("precomputed/precomputed_logvar_patientspace.npy")
precomputed_labels = np.load("precomputed/precomputed_labels.npy")

In [4]:
mu_subject = np.load("test/patientspace_mu.npy")
logvar_subject = np.load("test/patientspace_logvar.npy")

In [5]:
full_graph = np.zeros((len(patientspace_mu), len(patientspace_mu)))
for i in tqdm(range(len(patientspace_mu))):
    for j in range(len(patientspace_mu)):
        # M[i, j] = wasserstein(mu_zy[i], logvar_zy[i], mu_zy[j], logvar_zy[j])
        full_graph[i, j] = 0.5 * (KL_DIV((patientspace_mu[i]), (patientspace_logvar[i]), (patientspace_mu[j]), (patientspace_logvar[j])) +\
                         KL_DIV((patientspace_mu[j]), (patientspace_logvar[j]), (patientspace_mu[i]), (patientspace_logvar[i])))

  0%|          | 0/946 [00:00<?, ?it/s]

100%|██████████| 946/946 [00:22<00:00, 41.69it/s]


### Predict for new subject

In [6]:
dissimilarity =  np.zeros((len(mu_subject), len(patientspace_mu)))
for i in tqdm(range(len(mu_subject))):
    for j in range(len(patientspace_mu)):
        dissimilarity[i, j] = 0.5 * (KL_DIV((mu_subject[i]), (mu_subject[i]), (patientspace_mu[j]), (patientspace_logvar[j])) + KL_DIV((patientspace_mu[j]), (patientspace_logvar[j]), (mu_subject[i]), (mu_subject[i])))

100%|██████████| 1/1 [00:00<00:00, 36.52it/s]


In [7]:
from sklearn.neighbors import NearestNeighbors

In [8]:
onehot_labels = np.eye(len(np.unique(precomputed_labels)))[precomputed_labels]

In [9]:
neigh  = NearestNeighbors(n_neighbors=13, metric = 'precomputed').fit(full_graph)
D, idx = neigh.kneighbors(dissimilarity, n_neighbors=11)

W = (1 / D) / np.sum(1 / D)
proba = np.dot(W, onehot_labels[idx.ravel(), :])

### Compare with model classifier

In [10]:
model_proba = np.load("test/patientspace_classifier_pred.npy")

In [12]:
# Assuming proba and model_proba are torch tensors or numpy arrays of shape [1, 3]
p_space = proba.squeeze()       # Convert [1, 3] to [3]
p_model = model_proba.squeeze() # Convert [1, 3] to [3]
diag_list = ["CN", "AD", "FTD"]

print(f"--- Class Probabilities ---")
for i in range(3):
    print(f"Class {diag_list[i]} -> PatientSpace: {p_space[i]:.4f} | Model: {p_model[i]:.4f}")

--- Class Probabilities ---
Class CN -> PatientSpace: 0.0000 | Model: 0.0024
Class AD -> PatientSpace: 0.9086 | Model: 0.9925
Class FTD -> PatientSpace: 0.0914 | Model: 0.0052
